# Lab 5A: Explain a model and check fairness (10 minutes)

**Presenter: Nagarjuna Naidu** | Day 5 | about 10 minutes

- Explain a loan-default model globally (which features matter) with SHAP
- Explain one declined applicant in plain English (an adverse-action reason)
- Check approval rates by group, and see what dropping a proxy feature does

## 1. Setup: synthetic loan data and a model
Data is generated in code (seed 42). `postcode_risk` is a **proxy**: it tracks the sensitive attribute `group`.

In [ ]:
import os
for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"): os.environ[k] = "1"
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, shap
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
INK, TEAL, LIME, CORAL, GOLD, PURPLE = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134", "#6C4AB6"

rng = np.random.default_rng(42); n = 4000
isB = rng.random(n) < 0.4
group = np.where(isB, "B", "A")
df = pd.DataFrame({
    "income": np.clip(rng.lognormal(10.9, 0.4, n) * np.where(isB, 0.9, 1), 15000, None).round(-2),
    "debt_ratio": np.clip(rng.normal(0.35, 0.12, n) + np.where(isB, 0.03, 0), 0.02, 0.9),
    "credit_history_years": np.clip(rng.gamma(3, 3, n), 0, 40),
    "loan_amount": np.clip(rng.lognormal(9.8, 0.5, n), 2000, None).round(-2),
    "employment_years": np.clip(rng.gamma(2.5, 2.5, n), 0, 35),
    "postcode_risk": np.clip(0.2 + 0.5 * isB + rng.normal(0, 0.15, n), 0, 1),   # proxy for group
})
z = (-1.2 + 4 * (df.debt_ratio - 0.35) - 1.8 * (np.log(df.income) - 10.9) + 0.9 * (np.log(df.loan_amount) - 9.8)
     - 0.06 * df.credit_history_years - 0.05 * df.employment_years + 0.8 * (df.postcode_risk - 0.35))
default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

Xtr, Xte, ytr, yte, gtr, gte = train_test_split(df, default, group, test_size=0.3, random_state=42, stratify=default)
def fit(cols):
    return GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42).fit(Xtr[cols], ytr)
model = fit(list(df.columns))
p_def = model.predict_proba(Xte)[:, 1]
print(f"{n} applicants, default rate {default.mean():.1%}, corr(postcode_risk, group B) = {np.corrcoef(df.postcode_risk, isB)[0,1]:.2f}")
print(f"Test AUC: {roc_auc_score(yte, p_def):.3f}")

## 2. Global explanation: what drives the model?
SHAP gives each feature a push (up or down) on each prediction. Mean |SHAP| over 500 test rows = overall importance.

In [ ]:
explainer = shap.TreeExplainer(model)
S = explainer.shap_values(Xte.iloc[:500])
imp = pd.Series(np.abs(S).mean(0), index=df.columns).sort_values()
ax = imp.plot.barh(color=[CORAL if c == "postcode_risk" else TEAL for c in imp.index], figsize=(5, 2.6))
ax.set_xlabel("mean |SHAP| (log-odds of default)"); ax.set_title("Global feature importance", color=INK)
plt.tight_layout(); plt.show()
print("Ranking:", " > ".join(imp.index[::-1]))

**What to notice:** `income` and `debt_ratio` lead, `loan_amount` is third, and the proxy `postcode_risk` still ranks 5th of 6, so the model does use a feature we would rather it did not.

## 3. Local explanation: why was this applicant declined?
Decision rule for the demo: decline if predicted default probability is 0.20 or higher.

In [ ]:
THR = 0.20
i = int(np.argmax(p_def[:500] >= THR))             # first declined applicant among the 500 explained rows
row = Xte.iloc[i]
contrib = pd.Series(S[i], index=df.columns).sort_values(key=abs, ascending=False)
print(f"Applicant #{Xte.index[i]}: predicted default probability {p_def[i]:.2f} -> DECLINED (threshold {THR})")
print(pd.DataFrame({"value": row[contrib.index].round(2), "contribution": contrib.round(3)}).to_string())
ax = contrib[::-1].plot.barh(color=[CORAL if v > 0 else LIME for v in contrib[::-1]], figsize=(5, 2.6))
ax.axvline(0, color=INK, lw=0.8); ax.set_xlabel("pushes default risk up (+) or down (-)"); plt.tight_layout(); plt.show()

WORDS = {"income": "income level", "debt_ratio": "debt-to-income ratio", "credit_history_years": "length of credit history",
         "loan_amount": "requested loan amount", "employment_years": "time in employment", "postcode_risk": "postcode risk score"}
top = [WORDS[c] for c in contrib.index if contrib[c] > 0.1][:3]      # only contributions that really push towards decline
print("\nReason string:", "Your application was declined mainly because of your " + ", ".join(top[:-1]) + (" and " if len(top) > 1 else "") + top[-1] + ".")

## 4. Fairness check: approval rates by group
Disparate impact ratio = lower approval rate / higher approval rate. The common rule of thumb flags anything below 0.8.
We also retrain **without** the proxy and compare.

In [ ]:
def check(cols, label):
    m = model if cols == list(df.columns) else fit(cols)
    p = m.predict_proba(Xte[cols])[:, 1]
    appr = pd.Series(p < THR).groupby(gte).mean()
    ratio = appr.min() / appr.max()
    return {"model": label, "approval A": appr["A"], "approval B": appr["B"], "DI ratio": ratio,
            "flag (<0.8)": "FLAG" if ratio < 0.8 else "ok", "AUC": roc_auc_score(yte, p)}
cols_all = list(df.columns)
tbl = pd.DataFrame([check(cols_all, "all features"), check([c for c in cols_all if c != "postcode_risk"], "drop postcode_risk")]).set_index("model")
print(tbl.round(3).to_string())

**Honest reading:** dropping the proxy changes the ratio and the AUC only slightly, because income and debt ratio also differ by group in this data, so the bias is not removed by deleting one column.
Fairness definitions conflict (equal approval rates vs equal error rates cannot all hold), so this is a screening check, not proof of fairness.
LIME is another local explainer (it fits a small model around one prediction); we do not run it here.